# NSU_DEMO Lakehouse CSV Loader
Attach `NSU_DEMO` as the notebook's **default Lakehouse** before running.
The notebook reads relative paths such as `Files/dimension_tables/dim_school.csv` and replaces every managed Delta table.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, DateType, DecimalType, IntegerType, StringType, StructField, StructType

SOURCE_ROOT = 'Files'
DIMENSION_FOLDER = 'dimension_tables'
MART_FOLDER = 'mart_tables'
GOVERNANCE_FOLDER = 'data_governance'
EXPECTED_COUNTS = {
    'dim_school': 12, 'dim_program': 36, 'dim_term': 6,
    'fact_enrollment': 2148, 'fact_recruitment_funnel': 300,
    'fact_census_enrollment': 1074, 'certification_catalog': 3,
    'lineage_summary': 3, 'quality_test_evidence': 47,
}
print('Default Lakehouse source root:', SOURCE_ROOT)

In [ ]:
school_schema = StructType([
    StructField('school_id', StringType(), False), StructField('school_name', StringType(), False),
    StructField('school_code', StringType(), False),
])
program_schema = StructType([
    StructField('program_id', StringType(), False), StructField('school_id', StringType(), False),
    StructField('program_name', StringType(), False), StructField('degree_level', StringType(), False),
    StructField('cip_code', StringType(), False),
])
term_schema = StructType([
    StructField('term_id', StringType(), False), StructField('term_name', StringType(), False),
    StructField('academic_year', StringType(), False), StructField('term_start_date', DateType(), False),
    StructField('census_date', DateType(), False), StructField('term_end_date', DateType(), False),
])
print('Dimension schemas defined')

In [ ]:
# Preflight source files using the attached default Lakehouse.
sources = {
    'dim_school': DIMENSION_FOLDER, 'dim_program': DIMENSION_FOLDER, 'dim_term': DIMENSION_FOLDER,
    'fact_enrollment': MART_FOLDER, 'fact_recruitment_funnel': MART_FOLDER,
    'fact_census_enrollment': MART_FOLDER,
    'certification_catalog': GOVERNANCE_FOLDER, 'lineage_summary': GOVERNANCE_FOLDER,
    'quality_test_evidence': GOVERNANCE_FOLDER,
}
for table_name, folder in sources.items():
    path = f'{SOURCE_ROOT}/{folder}/{table_name}.csv'
    # Do not call notebookutils.fs.exists() here; some Fabric runtimes
    # resolve it to /user/trusted-service-user and return OneLake 400.
    print('Expected source', path)

In [ ]:
# Create or replace every table from its uploaded CSV.
# Relative Files/... paths are intentional for a Fabric notebook with NSU_DEMO attached as default.
for table_name, folder in sources.items():
    path = f'{SOURCE_ROOT}/{folder}/{table_name}.csv'
    df = (spark.read.option('header', True).option('inferSchema', True).option('mode', 'FAILFAST').csv(path))
    if not df.columns:
        raise ValueError(f'{table_name}: CSV has no columns')
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table_name))
    actual = spark.table(table_name).count()
    expected = EXPECTED_COUNTS[table_name]
    if actual != expected:
        raise ValueError(f'{table_name}: expected {expected} rows, found {actual}')
    print(f'CREATE OR REPLACE complete: dbo.{table_name} ({actual} rows)')

## Validation
Use the SQL analytics endpoint after this notebook completes:
```sql
SELECT TABLE_SCHEMA, TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'dbo';
SELECT * FROM dbo.dim_school;
```
The Spark-managed tables are exposed through the SQL endpoint under the `dbo` schema.